# N1 trên Kaggle: xuất split và chạy thử eval

Chạy lần lượt từng ô trong một notebook Kaggle. Bật **Internet** (clone repo) và **GPU** (ô 6).
Gõ dấu `-` thường khi nhập đường dẫn, không copy từ PDF.
Kỳ vọng số dòng train: CUB 5.994, Cars 8.144, Aircraft 6.667.

## 1. Clone repo và kiểm tra môi trường

In [ ]:
REPO_URL = "https://github.com/tanq1965/gcd-text-leak.git"
BRANCH = "n1-splits-eval"
REPO = "/kaggle/working/gcd-text-leak"

!git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git branch --show-current
!git rev-parse HEAD
!python -c "import torch, pandas, sklearn, scipy, joblib, matplotlib; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 2. Xác nhận tên thư mục dữ liệu trong /kaggle/input

In [ ]:
import os
print(os.listdir('/kaggle/input'))

# Tìm các thư mục/file mốc để biết root của từng bộ dữ liệu
for name, kind in [("CUB_200_2011", "cub_root = thư mục CHA của thư mục này"),
                   ("devkit", "CARS_SRC = thư mục CHA của thư mục này (cùng cars_train, cars_test)"),
                   ("images_variant_trainval.txt", "aircraft_root = thư mục cha của thư mục 'data' chứa file này")]:
    print(f"--- {name}  ({kind})")
    for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
        if dirpath.count(os.sep) - '/kaggle/input'.count(os.sep) > 7:
            dirnames[:] = []
            continue
        if name in dirnames or name in filenames:
            print(os.path.join(dirpath, name))

## 3. Root dữ liệu, dựng thư mục Cars (link từng phần như N0), kiểm tra cấu trúc

Nếu ô 2 cho tên thư mục khác thì sửa 3 hằng số `CUB_ROOT`, `CARS_SRC`, `AIRCRAFT_ROOT`.

In [ ]:
import os

CUB_ROOT = "/kaggle/input/datasets/wenewone/cub2002011"
CARS_SRC = "/kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars"
AIRCRAFT_ROOT = "/kaggle/input/datasets/seryouxblaster764/fgvc-aircraft/fgvc-aircraft-2013b/fgvc-aircraft-2013b"
CAR_ROOT = "/kaggle/working/cars/stanford_cars"

# Cars: thư mục thật trong /kaggle/working, link từng phần từ CARS_SRC (theo N0)
assert os.path.isdir(CARS_SRC), f"không thấy CARS_SRC: {CARS_SRC}"
os.makedirs(f"{CAR_ROOT}/devkit", exist_ok=True)
for d in ["cars_train", "cars_test"]:
    if not os.path.lexists(f"{CAR_ROOT}/{d}"):
        os.symlink(f"{CARS_SRC}/{d}", f"{CAR_ROOT}/{d}")
for f in os.listdir(f"{CARS_SRC}/devkit"):
    if not os.path.lexists(f"{CAR_ROOT}/devkit/{f}"):
        os.symlink(f"{CARS_SRC}/devkit/{f}", f"{CAR_ROOT}/devkit/{f}")
lbl = f"{CAR_ROOT}/devkit/cars_test_annos_withlabels.mat"
if not os.path.lexists(lbl):
    os.symlink(f"{CARS_SRC}/cars_test_annos_withlabels.mat", lbl)

def need(root, *parts):
    p = os.path.join(root, *parts)
    assert os.path.exists(p), f"thiếu: {p}"

need(CUB_ROOT, "CUB_200_2011", "images.txt")
need(CUB_ROOT, "CUB_200_2011", "images")
need(CAR_ROOT, "devkit", "cars_train_annos.mat")
need(CAR_ROOT, "devkit", "cars_test_annos_withlabels.mat")   # get_scars_datasets dựng cả tập test
need(CAR_ROOT, "cars_train")
need(CAR_ROOT, "cars_test")
need(AIRCRAFT_ROOT, "data", "images_variant_trainval.txt")
need(AIRCRAFT_ROOT, "data", "images_variant_test.txt")
need(AIRCRAFT_ROOT, "data", "images")
print("cấu trúc thư mục: OK")
print("cars:", os.listdir(CAR_ROOT))

## 4. Xuất split 2 lần cho 3 tập, so md5

In [ ]:
import hashlib, pathlib, subprocess, sys

ROOTS = {"cub": CUB_ROOT, "scars": CAR_ROOT, "aircraft": AIRCRAFT_ROOT}
EXPORT = pathlib.Path("/kaggle/working/export")
logs = {}

for ds, root in ROOTS.items():
    for run in ("run1", "run2"):
        d = EXPORT / run
        d.mkdir(parents=True, exist_ok=True)
        r = subprocess.run(
            [sys.executable, "-m", "src.export_splits", "--dataset", ds, "--data-root", root,
             "--out", str(d / f"{ds}.csv"), "--classmap-out", str(d / f"{ds}_classmap.csv")],
            capture_output=True, text=True, cwd=REPO)
        logs[(ds, run)] = r
        print(ds, run, "returncode =", r.returncode)
        if r.returncode != 0:
            print(r.stdout[-3000:]); print(r.stderr[-3000:])

def md5(p):
    return hashlib.md5(pathlib.Path(p).read_bytes()).hexdigest()

print()
for ds in ROOTS:
    for name in (f"{ds}.csv", f"{ds}_classmap.csv"):
        a, b = (EXPORT / run / name for run in ("run1", "run2"))
        if a.exists() and b.exists():
            print("SAME     " if md5(a) == md5(b) else "DIFFERENT", name, md5(a), md5(b))
        else:
            print("MISSING  ", name)

## 5. Bảng tự kiểm và 5 dòng đầu của mỗi file

In [ ]:
import pandas as pd

for ds in ROOTS:
    print("=" * 30, ds)
    print(logs[(ds, "run1")].stdout)
    for name in (f"{ds}.csv", f"{ds}_classmap.csv"):
        p = EXPORT / "run1" / name
        if p.exists():
            print("---", name)
            print(pd.read_csv(p, keep_default_na=False).head(5).to_string())

## 6. Chạy thử `src.eval` trên CUB (đặc trưng NGẪU NHIÊN)

> **CHỈ để chạy thông dòng lệnh, KHÔNG phải kết quả thật.** Đặc trưng là nhiễu ngẫu nhiên chuẩn hoá L2,
> nên ACC chỉ ở mức ngẫu nhiên. Kết quả ghi vào thư mục tạm, không vào `results/`.
> Thứ tự dòng của file đặc trưng được xáo cố ý, để thử cả bước căn hàng theo `image_id`.

In [ ]:
import json
import numpy as np

tmp = pathlib.Path("/kaggle/working/tmp_n1")
tmp.mkdir(exist_ok=True)
splits_cub = pd.read_csv(EXPORT / "run1" / "cub.csv", dtype=str)
ids = splits_cub["image_id"].to_numpy()

rng = np.random.default_rng(0)
perm = rng.permutation(len(ids))
x = rng.standard_normal((len(ids), 512)).astype(np.float32)
x /= np.linalg.norm(x, axis=1, keepdims=True)
np.save(tmp / "cub_img_RANDOM.npy", x[perm])
(tmp / "cub_img_RANDOM.ids.txt").write_text("\n".join(ids[perm]) + "\n", encoding="utf-8")

out_json = tmp / "cub_RANDOM_TEST_s0.json"
!python -m src.eval --dataset cub --variant RANDOM_TEST --splits {EXPORT}/run1/cub.csv --feats {tmp}/cub_img_RANDOM.npy --seed 0 --device cuda --n-init 1 --max-iter 20 --out {out_json}

rec = json.loads(out_json.read_text())
print({k: v for k, v in rec.items() if k != "pred"})
print("len(pred) =", len(rec["pred"]), "| pred_scope =", rec["pred_scope"])

## 7. Nén splits để tải về

Chỉ copy bộ `run1` của những tập có cả hai file SAME ở ô 4.

In [ ]:
import shutil

(pathlib.Path(REPO) / "splits").mkdir(exist_ok=True)
for ds in ROOTS:
    for name in (f"{ds}.csv", f"{ds}_classmap.csv"):
        a, b = (EXPORT / run / name for run in ("run1", "run2"))
        if a.exists() and b.exists() and md5(a) == md5(b):
            shutil.copy(a, pathlib.Path(REPO) / "splits" / name)
        else:
            print("BỎ QUA (không khớp hoặc thiếu):", name)

print(shutil.make_archive("/kaggle/working/splits_n1", "zip", str(pathlib.Path(REPO) / "splits")))
print(sorted(os.listdir(pathlib.Path(REPO) / "splits")))

Tải `/kaggle/working/splits_n1.zip` từ tab Output rồi commit `splits/*.csv` từ máy cá nhân.